# FHIR Query Validator Factory - Loop Engineering Demo

This notebook demonstrates the feedback loops implemented in the system:

1. **Normal valid query** — cache → validate → execute
2. **Learner escalation** — repeated invalid queries (3+ in 10 minutes)
3. **Server switcher** — same query against multiple public FHIR servers
4. **Human escalation** — repeated invalid queries (5+ in 15 minutes) with pause/resume

In [ ]:
import sys
import os

sys.path.append(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath("__file__")))))

from src.agentic_layer.graph.validation_workflow import run_validation_workflow
from src.agentic_layer.graph.workflow_engine import human_gate
from src.agentic_layer.config.settings import DEFAULT_SERVERS


def print_summary(result: dict, *, label: str = "") -> None:
    """Print a compact view of workflow final_output."""
    out = result.get("final_output", {})
    if label:
        print(f"\n--- {label} ---")
    print(f"  server_used           : {out.get('server_used')}")
    print(f"  valid                 : {out.get('valid')}")
    print(f"  executed              : {out.get('executed')}")
    print(f"  pattern_detected      : {out.get('pattern_detected')}")
    print(f"  escalation            : {out.get('escalation')}")
    print(f"  human_review_required : {out.get('human_review_required')}")
    if out.get("errors"):
        print(f"  errors                : {out['errors'][:2]}")
    if result.get("learner_guidance"):
        print(f"  learner_message       : {result['learner_guidance'].get('message')}")
    if out.get("human_review"):
        review = out["human_review"]
        print(f"  review_id             : {review.get('review_id')}")
        print(f"  severity              : {review.get('severity')}")

## Scenario 1: Normal Valid Query

In [ ]:
state = {
    "query_url": "Patient?gender=male",
    "server_key": "hapi",
    "user_id": "user-alice",
    "mode": "validate_and_execute",
}

result = run_validation_workflow(state)
print_summary(result, label="Scenario 1")

## Scenario 2: Repeated Invalid Queries (Triggers Learner)

Threshold: **3+ invalid queries within 10 minutes** → `escalation: learner`

In [ ]:
for i in range(1, 4):
    print(f"\n--- Attempt {i} ---")
    state = {
        "query_url": "Patient?invalid_param=true",
        "server_key": "hapi",
        "user_id": "user-bob",
        "mode": "validate_and_execute",
    }
    result = run_validation_workflow(state)
    print_summary(result)

## Scenario 3: Server Switcher

Run the same query against each configured public test server. Each server has a different `CapabilityStatement`, so validation outcomes may differ.

Registered keys: `hapi`, `firely`, `spark`, `wildfhir` (see `src/agentic_layer/config/settings.py`).

In [ ]:
QUERY = "Patient?gender=male"
PUBLIC_SERVERS = ["hapi", "firely", "spark", "wildfhir"]

print("Configured servers:")
for key in PUBLIC_SERVERS:
    info = DEFAULT_SERVERS[key]
    print(f"  {key:10} → {info['name']} ({info['base_url']})")

print("\nRunning validate_only across servers (live HTTP to /metadata):")
for server_key in PUBLIC_SERVERS:
    result = run_validation_workflow({
        "query_url": QUERY,
        "server_key": server_key,
        "user_id": "notebook-server-switcher",
        "mode": "validate_only",
    })
    print_summary(result, label=server_key)

In [ ]:
# Optional: execute on a single server after comparing validation
result = run_validation_workflow({
    "query_url": QUERY,
    "server_key": "firely",
    "user_id": "notebook-server-switcher",
    "mode": "validate_and_execute",
})
print_summary(result, label="firely validate_and_execute")

## Scenario 4: Human Escalation

Threshold: **5+ invalid queries within 15 minutes** → `escalation: human` and automated processing is **paused** for that user until a reviewer resolves the case.

In [ ]:
HUMAN_USER = "user-charlie-human"
INVALID_QUERY = "Patient?invalid_param=true"

last_result = None
for i in range(1, 6):
    print(f"\n--- Human escalation attempt {i} ---")
    last_result = run_validation_workflow({
        "query_url": INVALID_QUERY,
        "server_key": "hapi",
        "user_id": HUMAN_USER,
        "mode": "validate_only",
    })
    print_summary(last_result)

assert last_result["final_output"]["escalation"] == "human"
assert last_result["final_output"]["human_review_required"] is True
print("\n✓ Human escalation triggered as expected.")

In [ ]:
# While paused, new requests for the same user are blocked
blocked = run_validation_workflow({
    "query_url": "Patient?gender=male",
    "server_key": "hapi",
    "user_id": HUMAN_USER,
    "mode": "validate_only",
})
print_summary(blocked, label="blocked while paused")
assert blocked["final_output"]["valid"] is False
assert "paused" in blocked["final_output"]["errors"][0].lower()

In [ ]:
# Resolve the human review and resume automated processing
review_id = last_result["final_output"]["human_review"]["review_id"]

resolution = human_gate.submit_review_decision(
    review_id,
    reviewer="notebook-operator",
    decision="continue_monitoring",
    rationale="Demo resolution — allow user to continue with monitoring.",
)
print(f"Review resolved: {resolution['review']['decision']}")
print(f"User resumed   : {resolution['resumed']}")

resumed = run_validation_workflow({
    "query_url": "Patient?gender=male",
    "server_key": "hapi",
    "user_id": HUMAN_USER,
    "mode": "validate_only",
})
print_summary(resumed, label="after human resolution")
assert resumed["final_output"]["valid"] is True